# 예제 02. 풀링의 역할
빅데이터프로그래밍 · 8주차

## 목표
- Max Pooling이 하는 일을 숫자로 확인한다
- 크기가 절반으로 줄어드는 것을 본다
- 조금 움직인 이미지에도 결과가 비슷하게 나오는 것을 확인한다


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)


## 1. 손으로 계산해 보기
2×2 구역마다 가장 큰 값 하나만 남깁니다.


In [ ]:
x = torch.tensor([
    [1., 3., 2., 4.],
    [5., 6., 1., 2.],
    [7., 8., 3., 0.],
    [1., 2., 9., 4.],
]).reshape(1, 1, 4, 4)

print("입력 4x4:"); print(x.squeeze())

pooled = F.max_pool2d(x, kernel_size=2)
print("\nMax Pool 2x2 → 2x2:"); print(pooled.squeeze())
print("\n왼쪽 위 구역 [1,3,5,6] 중 최댓값:", 6.0)


In [ ]:
# 평균 풀링도 있습니다 — 요즘은 Max를 더 많이 씁니다
print("Average Pool:"); print(F.avg_pool2d(x, 2).squeeze())


## 2. 크기가 절반으로 줄어듭니다
계산량이 4분의 1이 됩니다. 파라미터는 없습니다 — 학습할 값이 없는 층입니다.


In [ ]:
pool = nn.MaxPool2d(2)
for size in [28, 14, 7]:
    t = torch.randn(1, 1, size, size)
    print(f"{size:2d} x {size:2d}  →  {tuple(pool(t).shape[-2:])}")

print("\nMaxPool2d 파라미터 수:", sum(p.numel() for p in pool.parameters()))


## 3. 조금 움직여도 결과가 비슷합니다
CNN이 위치 변화에 덜 민감한 이유입니다.


In [ ]:
base = torch.zeros(1, 1, 8, 8)
base[0, 0, 2:4, 2:4] = 1.0                 # 작은 사각형

shifted = torch.zeros(1, 1, 8, 8)
shifted[0, 0, 2:4, 3:5] = 1.0              # 오른쪽으로 1칸

print("원본 풀링 결과:");  print(F.max_pool2d(base, 2).squeeze())
print("\n1칸 이동 후:"); print(F.max_pool2d(shifted, 2).squeeze())
print("\n두 결과가 같은가:", torch.equal(F.max_pool2d(base, 2), F.max_pool2d(shifted, 2)))


## 4. 실제 이미지에서 보기
풀링을 반복하면 세부는 사라지고 큰 모양만 남습니다.


In [ ]:
from torchvision import datasets, transforms

fm = datasets.FashionMNIST("./data", train=True, download=True,
                           transform=transforms.ToTensor())
img, _ = fm[0]
x = img.unsqueeze(0)

stages = [("원본 28", x)]
cur = x
for i in range(3):
    cur = F.max_pool2d(cur, 2)
    stages.append((f"pool {i+1}회 · {cur.shape[-1]}", cur))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (title, t) in zip(axes, stages):
    ax.imshow(t.squeeze(), cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


## 5. Conv와 Pool을 번갈아 쌓기
합성곱으로 특징을 찾고, 풀링으로 크기를 줄입니다. CNN의 기본 리듬입니다.


In [ ]:
block = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
)

h = x
print(f"{'입력':22s} {tuple(h.shape)}")
for layer in block:
    h = layer(h)
    print(f"{layer.__class__.__name__:22s} {tuple(h.shape)}")


채널은 늘고(1 → 16 → 32) 크기는 줄어듭니다(28 → 14 → 7). CNN의 전형적인 모양입니다.


## 직접 해보기
1. `MaxPool2d(4)` 를 쓰면 28은 몇이 되나요?
2. 위 `block` 에 세 번째 Conv+Pool을 추가하면 최종 shape은 얼마인가요?


In [ ]:
# 여기에 작성하세요
